In [1]:
import xml.etree.ElementTree as ET
from pathlib import Path
import json
import re
import random
import pandas as pd
import ast
from form990_parser import (
    parse_header,
    parse_return,
    get_org_type,
    get_return_type,
    NAMESPACE
)

In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
xml_data_directory = Path(r'xml')

In [82]:
def get_year_from_dir(year_dir):
    return str(year_dir).replace('xml\\', '')

def get_period_from_dir(period_dir, year_dir):
    return str(period_dir).replace(str(year_dir)+ '\\', '')

def exact_match(root, match_list, target):
    try:
        match_list.append(root.find(f'.//{target}').text)
    except AttributeError:
        pass

def partial_tag_match(root, match_list, target):
    for elem in root.iter():
        tag = elem.tag.replace(NAMESPACE, '')
        if target in tag.lower():
            match_list.append(tag)

def element_search(target, match_type, root_dir=Path('xml'), files_per_dir = 250, debug = True):
    matches_by_year = {}
    matches = []
    files = 0
    for year_dir in root_dir.iterdir():
        # Create key for dict
        year = get_year_from_dir(year_dir)
        matches_by_year[year] = {}
        for period_dir in year_dir.iterdir():
            # TODO: Remove zip files from xml folders
            if period_dir.suffix in ('.zip', '.csv'):
                continue
            counter = 0
            period_matches = []
            # Create key for sub-dict
            period = get_period_from_dir(period_dir, year_dir)
            for xml_file in period_dir.glob('*.xml'):
                # Arbitrary limit
                if counter >= files_per_dir:
                    matches_by_year[year][period] = period_matches
                    matches += period_matches
                    break
                counter += 1
                files += 1

                root = ET.parse(xml_file).getroot()
                match_type(root, period_matches, target)
    if debug:
        print(f'We have searched {files:,} files ({counter} per subdirectory) and have made {len(matches):,} partial matches.')

    return matches, matches_by_year

We need to have a better understanding of how the form types (e.g., 990, 990-PF, etc.) change the overall format of the xml file. We have also found variations in the element tags over time, so we are doing a partial mapping on hundreds of files to determine if the `<ReturnTypeCd>` element remains consistent across time.

In [47]:
matches, matches_by_year = element_search('return', partial_tag_match)

We have searched 7,500 files (250 per subdirectory) and have made 50,951 partial matches.


Of the matches found, below is a list of unique matches:

In [48]:
for match in set(matches):
    print(match)

Return
OriginalReturnTaxPaidAmt
StatesWhereCopyOfReturnIsFldCd
ReturnData
OriginalReturnOverpaymentAmt
InitialReturnFormerPubChrtyInd
MinimumInvestmentReturnGrp
EmploymentTaxReturnsFiledInd
ReturnsAndAllowancesAmt
ReturnHeader
InitialReturnInd
ReturnTs
MinimumInvestmentReturnAmt
AmendedReturnInd
FinalReturnInd
ReturnTypeCd
GroupReturnForAffiliatesInd


Given that there are no minor variations of the `ReturnTypeCd` variation, we can be fairly sure that it remains consistent.

We can now use this to check the full list of return types across a similar number of files.

In [49]:
filing_types, filing_type_by_year = element_search(f'{NAMESPACE}ReturnTypeCd', exact_match)

We have searched 7,500 files (250 per subdirectory) and have made 7,500 partial matches.


Unsurprisingly, we see that there are the four versions of the return type:

* `990`
* `990EZ`
* `990PF`
* `990-T`

In [51]:
print(set(filing_types))
for filing_year, filing_periods in filing_type_by_year.items():
    print(filing_year)
    for filing_period, filing_types in filing_periods.items():
        print('\t', filing_period)
        print('\t\t', set(filing_types))

{'990T', '990EZ', '990', '990PF'}
2019
	 2019_01A
		 {'990EZ', '990', '990PF'}
	 2019_02A
		 {'990EZ', '990', '990PF'}
	 2019_03A
		 {'990'}
2020
	 2020_02A
		 {'990EZ', '990', '990PF'}
	 2020_04A
		 {'990'}
	 2020_3A
		 {'990EZ', '990'}
2021
	 2021_01A
		 {'990EZ', '990', '990PF'}
2022
	 2022_01A
		 {'990T', '990EZ', '990', '990PF'}
2023
2024
	 2024_01A
		 {'990T', '990EZ', '990', '990PF'}
	 2024_02A
		 {'990EZ', '990', '990PF'}
2025
	 2025_01A
		 {'990T', '990EZ', '990', '990PF'}
	 2025_02A
		 {'990T', '990EZ', '990', '990PF'}
	 2025_03A
		 {'990T', '990EZ', '990', '990PF'}
	 2025_04A
		 {'990T', '990EZ', '990', '990PF'}
	 2025_05A
		 {'990T', '990EZ', '990PF'}
	 2025_05B
		 {'990'}
	 2025_06A
		 {'990T', '990EZ', '990', '990PF'}
	 2025_07A
		 {'990T', '990EZ', '990', '990PF'}
	 2025_08A
		 {'990T', '990EZ', '990', '990PF'}
	 2025_09A
		 {'990T', '990EZ', '990', '990PF'}
	 2025_10A
		 {'990T', '990EZ', '990', '990PF'}
	 2025_11A
		 {'990T', '990EZ', '990', '990PF'}
	 2025_11B
		 {'99

We next need to see how the form type affects the formatting of the xml files. We will first inspect how it affects how the organization type is recorded:

In [52]:
filing_types = set(filing_types)
exact_target = f'{NAMESPACE}ReturnTypeCd'
partial_target = 'org'
matches_by_year = {}
matches = []
for year_dir in xml_data_directory.iterdir():
    # Create key for dict
    year = get_year_from_dir(year_dir)
    matches_by_year[year] = {}
    for period_dir in year_dir.iterdir():
        # TODO: Remove zip files from xml folders
        if period_dir.suffix in ('.zip', '.csv'):
            continue
        period_partial_matches = []
        period_filing_types = []
        # Create key for sub-dict
        period = get_period_from_dir(period_dir, year_dir)
        print(period)
        matches_by_year[year][period] = {}
        for i, xml_file in enumerate(period_dir.glob('*.xml')):

            if set(period_filing_types) == filing_types or i == 50:
                matches_by_year[year][period]['filing_types'] = period_filing_types
                matches_by_year[year][period]['partial_matches'] = period_partial_matches
                matches += period_partial_matches
                break
            
            root = ET.parse(xml_file).getroot()
            partial_tag_match(root, period_partial_matches, partial_target)
            exact_match(root, period_filing_types, exact_target)

2019_01A
2019_02A
2019_03A
2020_02A
2020_04A
2020_1A
2020_3A
2021_01A
2022_01A
2024_01A
2024_02A
2025_01A
2025_02A
2025_03A
2025_04A
2025_05A
2025_05B
2025_06A
2025_07A
2025_08A
2025_09A
2025_10A
2025_11A
2025_11B
2025_11C
2025_11D
2025_12A
2026_1A
2026_2A
2026_3A
2026_4A


In [53]:
for filing_year, filing_periods in matches_by_year.items():
    print(filing_year)
    for filing_period, filing_types in filing_periods.items():
        for filing_type, partial_matches in filing_types.items():
            print('\t', filing_type)
            for partial_match in set(partial_matches):
                print('\t\t', partial_match)

2019
	 filing_types
		 990EZ
		 990
		 990PF
	 partial_matches
		 DeferredCompensationFlngOrgAmt
		 CompensationBasedOnRltdOrgsAmt
		 GrantsToDomesticOrgsGrp
		 OtherAssetsOrgGrp
		 PerformOfServicesForOthOrgInd
		 OtherCompensationRltdOrgsAmt
		 BaseCompensationFilingOrgAmt
		 OrganizationFollowsSFAS117Ind
		 LoansOrGuaranteesFromOthOrgInd
		 NontaxableBenefitsFilingOrgAmt
		 BonusFilingOrganizationAmount
		 CompReportPrior990FilingOrgAmt
		 TransferToOtherOrgInd
		 PctSect4940eOrgNetInvstIncmAmt
		 ControlledOrganizationInd
		 GrantsOtherOrganizationsInd
		 DivRelatedOrganizationInd
		 BusinessRlnWithOrgMemInd
		 OrgReportOrRegisterStateCd
		 RentalOfFacilitiesToOthOrgInd
		 Total501c3OrgCnt
		 OrganizationFiled990TInd
		 TypeOfOrganizationOtherInd
		 BonusRelatedOrganizationsAmt
		 AssetPurchaseFromOtherOrgInd
		 TaxRevLeviedOrgnztnlBnft509Grp
		 Form990OfOtherOrganizationsInd
		 TotalCompensationFilingOrgAmt
		 CompReportPrior990RltdOrgsAmt
		 SupportedOrgSectionC456Ind
		 Reimburs

In [54]:
set(matches)

{'ActivitiesEngagedOrgInvlmntInd',
 'AssetPurchaseFromOtherOrgInd',
 'AssetSaleToOtherOrgInd',
 'AverageHoursPerWeekRltdOrgRt',
 'BaseCompensationFilingOrgAmt',
 'BonusFilingOrganizationAmount',
 'BonusRelatedOrganizationsAmt',
 'BusinessRlnWithOrgMemInd',
 'CYDistriAttentiveSuprtOrgAmt',
 'ChangeToOrgDocumentsInd',
 'ChgMadeToOrgnzngDocNotRptInd',
 'CntrctWith3rdPrtyForGameRevInd',
 'CollegeOrganizationInd',
 'CompBasedOnRevenueOfFlngOrgInd',
 'CompBsdNetEarnsFlngOrgInd',
 'CompBsdNetEarnsRltdOrgsInd',
 'CompBsdOnRevRelatedOrgsInd',
 'CompReportPrior990FilingOrgAmt',
 'CompReportPrior990RltdOrgsAmt',
 'CompensationBasedOnRltdOrgsAmt',
 'ControlledOrganizationInd',
 'DeferredCompRltdOrgsAmt',
 'DeferredCompensationFlngOrgAmt',
 'DisclosedOrgLegCtrlInd',
 'DivRelatedOrganizationInd',
 'DomesticOrgMeetingSect4940eInd',
 'EndowmentsHeldRelatedOrgInd',
 'EndowmentsHeldUnrelatedOrgInd',
 'ExemptFrgnOrgTaxWthldAtSrceAmt',
 'ExmptCtrlOrgDeductionAmt',
 'ExmptCtrlOrgNetUnrltIncmAmt',
 'ExmptCt

Collect `N` random samples of each filing type across each period along with the partial matches for different flags:

In [94]:
partial_targets = [
    'org',
    'org.+ind',
    '527',
    'lobby'
]

filing_types = ['990EZ', '990', '990PF', '990T']

N = 25

In [95]:


output_csv = 'exact_and_partial_org_type_matches.csv'

for year_dir in xml_data_directory.iterdir():
    # Create key for dict
    year = get_year_from_dir(year_dir)
    matches_by_year[year] = {}
    for period_dir in year_dir.iterdir():
        # TODO: Remove zip files from xml folders
        if period_dir.suffix in ('.zip', '.csv'):
            continue

        # Create key for sub-dict
        period = get_period_from_dir(period_dir, year_dir)
        print(period)

        xml_files = [f for f in period_dir.glob('*.xml')]
        xml_files_random = random.sample(xml_files, k=len(xml_files))

        content = {
            'file': '',
            'year': year,
            'period': period,
            'ReturnTypeCd': '',
            'org': [],
            'org.+ind': [],
            '527': [], 
            'lobby': []
        }

        return_counts = {
            '990': 0,
            '990T': 0,
            '990PF': 0,
            '990EZ': 0
        }

        # for partial_target in partial_targets:
        #     content[partial_target] = []
        
        data = [
            {
                'file': '',
                'year': '',
                'period': '',
                'ReturnTypeCd': '',
                'org': [],
                'org.+ind': [],
                '527': [], 
                'lobby': []
            }
        ]

        for i, random_xml_file in enumerate(xml_files_random):
            if i % 1_000 == 0:
                print(i)
            if sum([return_count for return_count in return_counts.values()]) >= N * 4:
                break
            if i > 15_000:
                break
            # Get root
            root = ET.parse(random_xml_file).getroot()

            # Get basic info for categorization
            return_type_cd = root.find(f'.//{NAMESPACE}ReturnTypeCd').text
            if return_counts[return_type_cd] >= N:
                continue

            # Otherwise, collect the data
            content['file'] = str(random_xml_file)
            content['ReturnTypeCd'] = return_type_cd
            for partial_target in partial_targets:
                for elem in root.iter():
                    tag = elem.tag.replace(NAMESPACE, '')
                    if re.match(partial_target, tag, re.IGNORECASE):
                        content[partial_target].append(tag)
                        
            return_counts[return_type_cd] += 1

            data.append(content)
            
            if len(data) >= 50:
                df_to_output = pd.DataFrame(data)
                df_to_output.to_csv(
                    output_csv,
                    index=False,
                    mode='a'
                )
                del df_to_output
                data = [
                    {
                        'file': '',
                        'year': '',
                        'period': '',
                        'ReturnTypeCd': '',
                        'org': [],
                        'org.+ind': [],
                        '527': [], 
                        'lobby': []
                    }
                ]
            df_to_output = pd.DataFrame(data)
            df_to_output.to_csv(
                output_csv,
                index=False,
                mode='a'
            )
            del df_to_output

2019_01A
0
1000
2000
3000
4000
5000
6000
7000
8000
9000
10000
11000
12000
13000
14000
15000
2019_02A
0
1000
2000
3000
4000
5000
6000
7000
8000
9000
10000
11000
12000
13000
14000
15000
2019_03A
0
1000
2000
3000
4000
5000
6000
7000
8000
9000
10000
11000
12000
13000
14000
15000
2020_02A
0
1000
2000
3000
4000
5000
6000
7000
8000
9000
10000
11000
12000
13000
14000
15000
2020_03A
0
1000
2000
3000
4000
5000
6000
7000
8000
9000
10000
11000
12000
13000
14000
15000
2020_04A
0
1000
2000
3000
4000
5000
6000
7000
8000
9000
10000
11000
12000
13000
14000
15000
2020_1A
2021_01A
0
1000
2000
2022_01A
0
2023_01A
0
2023_02A
2024_01A
0
1000
2000
3000
4000
5000
6000
7000
8000
9000
10000
11000
12000
13000
14000
15000
2024_02A
0
1000
2000
3000
4000
5000
6000
7000
8000
9000
10000
11000
12000
13000
14000
15000
2025_01A
0
1000
2025_02A
0
2025_03A
0
2025_04A
0
1000
2025_05A
0
1000
2025_05B
0
1000
2025_06A
0
2025_07A
0
2025_08A
0
2025_09A
0
1000
2025_10A
0
1000
2025_11A
0
2025_11B
0
2025_11C
0
2025_11D
0
2025_12A


In [71]:
df = pd.DataFrame(data).iloc[1:]
df

,file,year,period,ReturnTypeCd,org,org.+ind,organization,organization.+ind
1,xml\2019\2019_01A\201901449349100500_public.xml,2019,2019_01A,990PF,"[Organization501c3ExemptPFInd, OrgDoesNotFollo...","[Organization501c3ExemptPFInd, OrgDoesNotFollo...","[Organization501c3ExemptPFInd, OrganizationDis...","[Organization501c3ExemptPFInd, OrganizationDis..."
2,xml\2019\2019_01A\201901449349100500_public.xml,2019,2019_01A,990PF,"[Organization501c3ExemptPFInd, OrgDoesNotFollo...","[Organization501c3ExemptPFInd, OrgDoesNotFollo...","[Organization501c3ExemptPFInd, OrganizationDis...","[Organization501c3ExemptPFInd, OrganizationDis..."
3,xml\2019\2019_01A\201901449349100500_public.xml,2019,2019_01A,990PF,"[Organization501c3ExemptPFInd, OrgDoesNotFollo...","[Organization501c3ExemptPFInd, OrgDoesNotFollo...","[Organization501c3ExemptPFInd, OrganizationDis...","[Organization501c3ExemptPFInd, OrganizationDis..."
4,xml\2019\2019_01A\201901449349100500_public.xml,2019,2019_01A,990PF,"[Organization501c3ExemptPFInd, OrgDoesNotFollo...","[Organization501c3ExemptPFInd, OrgDoesNotFollo...","[Organization501c3ExemptPFInd, OrganizationDis...","[Organization501c3ExemptPFInd, OrganizationDis..."
5,xml\2019\2019_01A\201901449349100500_public.xml,2019,2019_01A,990PF,"[Organization501c3ExemptPFInd, OrgDoesNotFollo...","[Organization501c3ExemptPFInd, OrgDoesNotFollo...","[Organization501c3ExemptPFInd, OrganizationDis...","[Organization501c3ExemptPFInd, OrganizationDis..."
...,...,...,...,...,...,...,...,...
1116,xml\2026\2026_4A\202621119339300402_public.xml,2026,2026_4A,990T,"[Organization501c3Ind, OrganizationFollowsFASB...","[Organization501c3Ind, OrganizationFollowsFASB...","[Organization501c3Ind, OrganizationFollowsFASB...","[Organization501c3Ind, OrganizationFollowsFASB..."
1117,xml\2026\2026_4A\202621119339300402_public.xml,2026,2026_4A,990T,"[Organization501c3Ind, OrganizationFollowsFASB...","[Organization501c3Ind, OrganizationFollowsFASB...","[Organization501c3Ind, OrganizationFollowsFASB...","[Organization501c3Ind, OrganizationFollowsFASB..."
1118,xml\2026\2026_4A\202621119339300402_public.xml,2026,2026_4A,990T,"[Organization501c3Ind, OrganizationFollowsFASB...","[Organization501c3Ind, OrganizationFollowsFASB...","[Organization501c3Ind, OrganizationFollowsFASB...","[Organization501c3Ind, OrganizationFollowsFASB..."
1119,xml\2026\2026_4A\202621119339300402_public.xml,2026,2026_4A,990T,"[Organization501c3Ind, OrganizationFollowsFASB...","[Organization501c3Ind, OrganizationFollowsFASB...","[Organization501c3Ind, OrganizationFollowsFASB...","[Organization501c3Ind, OrganizationFollowsFASB..."


In [72]:
df.to_csv(
    'exact_and_partial_org_type_matches.csv',
    index=False
)

In [73]:
df = pd.read_csv(
    'exact_and_partial_org_type_matches.csv',
    converters={
        'org': ast.literal_eval,
        'org.+ind': ast.literal_eval,
        'organization': ast.literal_eval,
        'organization.+ind': ast.literal_eval,
    }
)
df

,file,year,period,ReturnTypeCd,org,org.+ind,organization,organization.+ind
0,xml\2019\2019_01A\201901449349100500_public.xml,2019,2019_01A,990PF,"[Organization501c3ExemptPFInd, OrgDoesNotFollo...","[Organization501c3ExemptPFInd, OrgDoesNotFollo...","[Organization501c3ExemptPFInd, OrganizationDis...","[Organization501c3ExemptPFInd, OrganizationDis..."
1,xml\2019\2019_01A\201901449349100500_public.xml,2019,2019_01A,990PF,"[Organization501c3ExemptPFInd, OrgDoesNotFollo...","[Organization501c3ExemptPFInd, OrgDoesNotFollo...","[Organization501c3ExemptPFInd, OrganizationDis...","[Organization501c3ExemptPFInd, OrganizationDis..."
2,xml\2019\2019_01A\201901449349100500_public.xml,2019,2019_01A,990PF,"[Organization501c3ExemptPFInd, OrgDoesNotFollo...","[Organization501c3ExemptPFInd, OrgDoesNotFollo...","[Organization501c3ExemptPFInd, OrganizationDis...","[Organization501c3ExemptPFInd, OrganizationDis..."
3,xml\2019\2019_01A\201901449349100500_public.xml,2019,2019_01A,990PF,"[Organization501c3ExemptPFInd, OrgDoesNotFollo...","[Organization501c3ExemptPFInd, OrgDoesNotFollo...","[Organization501c3ExemptPFInd, OrganizationDis...","[Organization501c3ExemptPFInd, OrganizationDis..."
4,xml\2019\2019_01A\201901449349100500_public.xml,2019,2019_01A,990PF,"[Organization501c3ExemptPFInd, OrgDoesNotFollo...","[Organization501c3ExemptPFInd, OrgDoesNotFollo...","[Organization501c3ExemptPFInd, OrganizationDis...","[Organization501c3ExemptPFInd, OrganizationDis..."
...,...,...,...,...,...,...,...,...
1115,xml\2026\2026_4A\202621119339300402_public.xml,2026,2026_4A,990T,"[Organization501c3Ind, OrganizationFollowsFASB...","[Organization501c3Ind, OrganizationFollowsFASB...","[Organization501c3Ind, OrganizationFollowsFASB...","[Organization501c3Ind, OrganizationFollowsFASB..."
1116,xml\2026\2026_4A\202621119339300402_public.xml,2026,2026_4A,990T,"[Organization501c3Ind, OrganizationFollowsFASB...","[Organization501c3Ind, OrganizationFollowsFASB...","[Organization501c3Ind, OrganizationFollowsFASB...","[Organization501c3Ind, OrganizationFollowsFASB..."
1117,xml\2026\2026_4A\202621119339300402_public.xml,2026,2026_4A,990T,"[Organization501c3Ind, OrganizationFollowsFASB...","[Organization501c3Ind, OrganizationFollowsFASB...","[Organization501c3Ind, OrganizationFollowsFASB...","[Organization501c3Ind, OrganizationFollowsFASB..."
1118,xml\2026\2026_4A\202621119339300402_public.xml,2026,2026_4A,990T,"[Organization501c3Ind, OrganizationFollowsFASB...","[Organization501c3Ind, OrganizationFollowsFASB...","[Organization501c3Ind, OrganizationFollowsFASB...","[Organization501c3Ind, OrganizationFollowsFASB..."


In [74]:
df.loc[:, 'ReturnTypeCd'].unique()

<StringArray>
['990PF', '990T']
Length: 2, dtype: str

In [75]:
targets = [
    'OrgDoesNotFollowFASB117Ind',
    'OrgDoesNotFollowSFAS117Ind',
    'OrgFiledInLieuOfForm1041Ind',
    'OrgProfitOrOwnershipPct',
    'OrgReportOrRegisterStateCd',
    'Organization4947a1NotPFInd',
    'Organization4947a1TrtdPFInd',
    'Organization501Ind',
    'Organization501IndicatorGrp',
    'Organization501c3ExemptPFInd',
    'Organization501c3Ind',
    'Organization501c3Name',
    'Organization501c3TaxablePFInd',
    'Organization501cCorporationInd',
    'Organization501cInd',
    'Organization501cTrustInd',
    'Organization501cTypeTxt',
    'OrganizationBelongsAffltGrpInd',
    'OrganizationBusinessName',
    'OrganizationChangeSuprtOrgInd',
    'OrganizationDissolvedEtcInd',
    'OrganizationFiled990TInd',
    'OrganizationFollowsFASB117Ind',
    'OrganizationFollowsSFAS117Ind',
    'OrganizationHadUBIInd',
    'OrganizationIncurExciseTaxInd',
    'OrganizationTypeCd',
    'OrganizationTypeDesc'
]

org_type_variant_breakdown = {}

all_matches = set()
for year in df.loc[:, 'year'].unique():
    year_df = df.loc[df.loc[:, 'year']==year]
    year_str = str(year)
    org_type_variant_breakdown[year_str] = {}
    for period in year_df.loc[:, 'period'].unique():
        period_df = year_df.loc[year_df.loc[:, 'period']==period]
        org_type_variant_breakdown[year_str][period] = {}
        # print(year, period)
        for return_type_cd in period_df.loc[:, 'ReturnTypeCd'].unique():
            # print(return_type_cd)
            return_type_df = period_df.loc[period_df.loc[:, 'ReturnTypeCd'] == return_type_cd]
            org_type_variant_breakdown[year_str][period][return_type_cd] = {}
            for partial_target in partial_targets:
                # print(partial_target)
                org_type_variant_breakdown[year_str][period][return_type_cd] = set()
                for partial_target_list in return_type_df.loc[:, partial_target].tolist():
                    org_type_variant_breakdown[year_str][period][return_type_cd] |= set(partial_target_list)
                    all_matches |= set(partial_target_list)
org_type_variant_breakdown

{'2019': {'2019_01A': {'990PF': {'Organization501c3ExemptPFInd',
    'Organization501c3Ind',
    'Organization501cInd',
    'OrganizationDissolvedEtcInd',
    'OrganizationFiled990TInd',
    'OrganizationFollowsSFAS117Ind',
    'OrganizationHadUBIInd'}},
  '2019_02A': {'990PF': {'Organization4947a1TrtdPFInd',
    'Organization501c3ExemptPFInd',
    'Organization501c3Ind',
    'Organization501cInd',
    'OrganizationChangeSuprtOrgInd',
    'OrganizationDissolvedEtcInd',
    'OrganizationFiled990TInd',
    'OrganizationFollowsSFAS117Ind',
    'OrganizationHadUBIInd'}},
  '2019_03A': {'990PF': {'Organization4947a1TrtdPFInd',
    'Organization501c3ExemptPFInd',
    'Organization501c3Ind',
    'Organization501cInd',
    'OrganizationDissolvedEtcInd',
    'OrganizationFiled990TInd',
    'OrganizationFollowsSFAS117Ind',
    'OrganizationHadUBIInd'}}},
 '2020': {'2020_02A': {'990PF': {'Organization4947a1TrtdPFInd',
    'Organization501c3ExemptPFInd',
    'Organization501c3Ind',
    'Organizati

In [1]:
df

NameError: name 'df' is not defined

In [87]:
element_search('Organization501cTypeTxt', exact_match, files_per_dir=150)

We have searched 4,500 files (150 per subdirectory) and have made 0 partial matches.


([],
 {'2019': {'2019_01A': [], '2019_02A': [], '2019_03A': []},
  '2020': {'2020_02A': [], '2020_03A': [], '2020_04A': []},
  '2021': {'2021_01A': []},
  '2022': {'2022_01A': []},
  '2023': {},
  '2024': {'2024_01A': [], '2024_02A': []},
  '2025': {'2025_01A': [],
   '2025_02A': [],
   '2025_03A': [],
   '2025_04A': [],
   '2025_05A': [],
   '2025_05B': [],
   '2025_06A': [],
   '2025_07A': [],
   '2025_08A': [],
   '2025_09A': [],
   '2025_10A': [],
   '2025_11A': [],
   '2025_11B': [],
   '2025_11C': [],
   '2025_11D': [],
   '2025_12A': []},
  '2026': {'2026_1A': [], '2026_2A': [], '2026_3A': [], '2026_4A': []}})

In [ ]:
for year_dir in xml_data_directory.iterdir():
    for period_dir in year_dir.iterdir():
        print(period_dir)
        counter = 0
        files_skipped = 0
        for xml_file in period_dir.glob('*.xml'):
            # Skip certain orgs and form types
            root = ET.parse(xml_file).getroot()
            return_type_cd = get_return_type(root)
            org_type = get_org_type(root, return_type_cd)
            
            if return_type_cd in ['990T'] or org_type in ['Other', '527']:
                print(str(xml_file), return_type_cd, org_type)

xml\2019\2019_01A
xml\2019\2019_02A
xml\2019\2019_03A
xml\2019\2019_TEOS_XML_CT1.zip
xml\2019\download990xml_2019_1.zip
xml\2019\download990xml_2019_2.zip
xml\2019\download990xml_2019_3.zip
xml\2019\download990xml_2019_4.zip
xml\2019\download990xml_2019_5.zip
xml\2019\download990xml_2019_6.zip
xml\2019\download990xml_2019_7.zip
xml\2019\download990xml_2019_8.zip
xml\2019\index_2019.csv
xml\2020\2020_02A
xml\2020\2020_03A
